# From notebook to a runnable script

Python stage, lesson P26. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** save notebook steps as a script, put them in functions under a main block, set parameters at the top, loop over the monthly files and write one output file.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the cafe-sales data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it.

## Setup

In [ ]:
import math
import numbers

import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Run this cell as it is. It also holds the answer checker used by the exercise cells.
EXERCISES = ["Exercise 1", "Exercise 2", "Exercise 3"]


RESULTS = {}


def same(got, expected):
    """True when two values agree: floats within rounding, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k]) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=1e-9)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps or {}).items():
        if same(got, wrong):
            print("Hint:", message)
    return False

def check_function(label, name, cases):
    """Call your function on every case. Each case is (arguments, keyword arguments, expected result)."""
    fn = globals().get(name)
    if not callable(fn):
        RESULTS[label] = None
        print(f"{label}: define a function called {name}, then run this cell again.")
        return False
    for args, kwargs, expected in cases:
        try:
            got = fn(*args, **kwargs)
        except Exception as error:
            RESULTS[label] = False
            print(f"{label}: not yet. {name}{args} stopped with {type(error).__name__}: {error}")
            return False
        if not same(got, expected):
            RESULTS[label] = False
            print(f"{label}: not yet. {name}{args} should give {expected!r}, but gave {got!r}.")
            if got is None:
                print("Hint: None means the function gave nothing back. Did it return the value, or only print it?")
            return False
    RESULTS[label] = True
    print(f"{label}: correct ({len(cases)} cases).")
    return True

## Data

In [ ]:
# The five cafe-sales tables (version 1), read from the release tag v0.1.0.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/"
stores = pd.read_csv(BASE + "stores.csv")
customers = pd.read_csv(BASE + "customers.csv")
products = pd.read_csv(BASE + "products.csv")
orders = pd.read_csv(BASE + "orders.csv")
order_items = pd.read_csv(BASE + "order_items.csv")

# Known answer from the dataset README: 3, 150, 16, 1,800 and 3,287 rows.
assert (len(stores), len(customers), len(products), len(orders), len(order_items)) == (3, 150, 16, 1800, 3287)
print("Data loaded: 3 stores, 150 customers, 16 products, 1,800 orders, 3,287 order lines.")
# The cafes send one orders file a month; the export is simulated: this cell cuts orders.csv into six files in a folder called monthly.
Path("monthly").mkdir(exist_ok=True)
month_of = pd.to_datetime(orders["ordered_at"]).dt.month
for m in range(1, 7):
    orders[month_of == m].to_csv(f"monthly/orders_2026_{m:02d}.csv", index=False)


def run_script(text, name="ex_script.py"):
    """Save `text` as a script, run it with this notebook's Python and give back what it printed."""
    Path(name).write_text(text)
    done = subprocess.run([sys.executable, name], capture_output=True, text=True)
    return (done.stdout + done.stderr).strip()

## Unit 1: Save a cell as a script and run it

**Predict first.** will this cell run the lines, or only save them? What will it print?

In [ ]:
%%writefile pack.py
import pandas as pd

orders = pd.read_csv("monthly/orders_2026_04.csv")
print(len(orders), "orders in April")

**Predict first.** the notebook's `orders` has 1,800 rows, and April has 349 orders. What will the script print, and what will `len(orders)` print afterwards?

In [ ]:
!python pack.py
print(len(orders))

## Unit 2: Functions and a main block

**Predict first.** this file already exists. What will the cell print?

In [ ]:
%%writefile pack.py
import pandas as pd


def count_orders(path):
    orders = pd.read_csv(path)
    return len(orders)


def main():
    print(count_orders("monthly/orders_2026_04.csv"), "orders in April")


if __name__ == "__main__":
    main()

**Predict first.** what does `import pack` itself print, and what will the other three lines give, in order?

In [ ]:
import pack
print("imported")
print(pack.count_orders("monthly/orders_2026_05.csv"))
!python pack.py

## Unit 3: Parameters at the top

**Predict first.** this file already exists. What will the cell print?

In [ ]:
%%writefile pack.py
import pandas as pd

# Parameters: the only lines to change between runs.
ORDER_FILE = "monthly/orders_2026_05.csv"
OUT_FILE = "orders_by_store.csv"


def orders_by_store(path):
    orders = pd.read_csv(path)
    return orders.groupby("store_id").size().rename("orders")


def main():
    table = orders_by_store(ORDER_FILE)
    table.to_csv(OUT_FILE)
    print("wrote", OUT_FILE, "with", len(table), "rows")


if __name__ == "__main__":
    main()

**Predict first.** May had 380 orders. What will the script print, and what will the file hold: which columns, and how many rows of numbers?

In [ ]:
!python pack.py
print(Path("orders_by_store.csv").read_text())

## Unit 4: Loop over the files, write one output

**Predict first.** this file already exists. What will the cell print?

In [ ]:
%%writefile pack.py
from pathlib import Path

import pandas as pd

# Parameters: the only lines to change between runs.
ORDER_FILES = "monthly/orders_2026_*.csv"
OUT_FILE = "orders_by_store_month.csv"


def read_orders(pattern):
    files = sorted(Path().glob(pattern))
    frames = [pd.read_csv(path) for path in files]
    return pd.concat(frames, ignore_index=True), len(files)


def main():
    orders, n_files = read_orders(ORDER_FILES)
    orders = orders[orders["status"] == "completed"]
    orders = orders.assign(month=pd.to_datetime(orders["ordered_at"]).dt.month)
    table = orders.groupby(["store_id", "month"]).size().rename("orders").reset_index()
    table.to_csv(OUT_FILE, index=False)
    print(n_files, "files,", len(orders), "orders,", len(table), "rows written to", OUT_FILE)


if __name__ == "__main__":
    main()

**Predict first.** the folder has 6 files and 3 stores. How many rows will the table have, and how many orders will the script report?

In [ ]:
!python pack.py

## Unit 5: Check: the script gives the notebook's table

**Predict first.** will the numbers the script wrote equal the numbers the notebook builds?

In [ ]:
expected = (orders.assign(month=pd.to_datetime(orders["ordered_at"]).dt.month)
            .query("status == 'completed'")
            .groupby(["store_id", "month"]).size().rename("orders").reset_index())
from_script = pd.read_csv("orders_by_store_month.csv")
print(from_script.to_numpy().tolist() == expected.to_numpy().tolist())

## Finish it

### Exercise 1: Finish it

**Task:** How many orders did one store place in one month? The function below answers for the store and month written at the top. Finish it: make `store_id` and `month` parameters of the function, so that the same function answers for any store and month.

**Expected:** `count_orders(orders, 3, 5)` gives the orders of Harbor, store 3, in May.

In [ ]:
STORE_ID = 2  # parameters at the top
MONTH = 4


def count_orders(orders):
    month = pd.to_datetime(orders["ordered_at"]).dt.month
    return int(((orders["store_id"] == STORE_ID) & (month == MONTH)).sum())

In [ ]:
_ = check_function("Exercise 1", "count_orders", [((orders, 2, 4), {}, 103), ((orders, 3, 5), {}, 121), ((orders, 1, 3), {}, 133)])

<details>
<summary>Hint</summary>

Add two parameters after `orders` and use them in the body instead of the two names at the top.

</details>

<details>
<summary>Solution</summary>

```python
def count_orders(orders, store_id, month):
    months = pd.to_datetime(orders["ordered_at"]).dt.month
    return int(((orders["store_id"] == store_id) & (months == month)).sum())
```

The function now takes the store and the month as arguments, so each call answers its own question. A function that kept reading `STORE_ID` and `MONTH` from the top would answer the same question for every call, and the check cell calls it for store 3 and May.

</details>

## On your own

### Exercise 2: On your own

**Task:** How many orders of Campus, store 2, were completed in March? Before you start, say in a sentence what the script prints. Write a script as text in a variable called `script`: the file name and the store go in capital names at the top, a `main()` function reads `monthly/orders_2026_03.csv` with pandas, keeps the completed orders of the store, and prints `N completed orders` with the number in place of N. Then run it with `answer_2 = run_script(script)`. Write the code in the empty cell of the notebook.

**Expected:** `answer_2` is the text `N completed orders` with the number of orders of store 2 that are completed.

In [ ]:
_ = check("Exercise 2", "answer_2", "124 completed orders", traps={"259 completed orders": "That counts the lines of the file, header included, for every store and status.", "125 completed orders": "That counts the orders of the store, canceled ones included."})

<details>
<summary>Hint</summary>

Read the file with `pd.read_csv`, keep the rows where the store is 2 and the status is `completed`, and print `len` of what is left with the words `completed orders`.

</details>

<details>
<summary>Solution</summary>

```python
script = """
import pandas as pd

ORDER_FILE = "monthly/orders_2026_03.csv"
STORE_ID = 2


def main():
    orders = pd.read_csv(ORDER_FILE)
    done = orders[(orders["store_id"] == STORE_ID) & (orders["status"] == "completed")]
    print(len(done), "completed orders")


if __name__ == "__main__":
    main()
"""
answer_2 = run_script(script)
```

The script prints `124 completed orders`. March has 125 orders of store 2 and one of them, order 1461, is canceled. Counting lines of the file gives 259, which is every order of every store plus the header, and the check cell says so; counting rows without the status filter gives 125.

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** Spot the flaw. A colleague's script writes the payment methods of the April orders to a file. It was meant to give the same file every time, but after it is run twice the file has twice as many lines. The cell saves the script, deletes the old file, runs the script twice and counts the lines of the file. Fix the script text so that `answer_3` is the number of lines of one correct file: a header and one line for each payment method.

**Expected:** `answer_3` is `4`.

In [ ]:
colleague_script = """
import pandas as pd

ORDER_FILE = "monthly/orders_2026_04.csv"
OUT_FILE = "payments.csv"


def main():
    orders = pd.read_csv(ORDER_FILE)
    table = orders.groupby("payment_method").size().rename("orders")
    table.to_csv(OUT_FILE, mode="a")


if __name__ == "__main__":
    main()
"""
Path("payments.csv").unlink(missing_ok=True)
run_script(colleague_script)
run_script(colleague_script)
answer_3 = len(Path("payments.csv").read_text().splitlines())

In [ ]:
_ = check("Exercise 3", "answer_3", 4, traps={8: "The file keeps growing. Look at the `mode` argument of `to_csv` in the script."})

<details>
<summary>Hint</summary>

Run the script text once, print the file, and run it a second time. What does `mode="a"` ask `to_csv` to do with a file that exists?

</details>

<details>
<summary>Solution</summary>

```python
colleague_script = """
import pandas as pd

ORDER_FILE = "monthly/orders_2026_04.csv"
OUT_FILE = "payments.csv"


def main():
    orders = pd.read_csv(ORDER_FILE)
    table = orders.groupby("payment_method").size().rename("orders")
    table.to_csv(OUT_FILE)


if __name__ == "__main__":
    main()
"""
Path("payments.csv").unlink(missing_ok=True)
run_script(colleague_script)
run_script(colleague_script)
answer_3 = len(Path("payments.csv").read_text().splitlines())
```

The file has 4 lines: the header and card, cash and mobile. `mode="a"` appends, so each run added its header and its three rows after the old ones and two runs gave 8 lines, which the check cell reports. `to_csv` writes a new file by default, which its page lists as mode `w`, so the same script gives the same file on every run.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")